# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mujahid1hm/flyrank-ai-/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose a logistic regression model as the primary learner for this lane because the task is a yes/no prediction problem with an observed label (`is_declining_label`), and the goal is to rank pages by risk in a way a human can read and trust. Logistic regression is easy to interpret, it handles the mixed numeric + categorical feature set cleanly, and it gives a probability score that matches the ranking goal much better than a hard label. I also keep the hand-rule baseline in the same comparison so the model has to earn its place.


In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

repo_root = Path(r"c:\Users\Mujahid\Documents\flyrank-ai-")
features = pd.read_csv(repo_root / "data" / "processed" / "refresh_feature_vector.csv")
baseline = pd.read_csv(repo_root / "data" / "processed" / "baseline_refresh_queue.csv")

numeric_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]
cat_cols = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier"
]

X_num = features[numeric_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
X_cat = features[cat_cols].fillna("unknown").astype(str)
X = pd.concat([
    X_num.reset_index(drop=True),
    pd.get_dummies(X_cat, prefix=cat_cols, dtype=float).reset_index(drop=True),
], axis=1)
y = features["is_declining_label"].astype(int)

clients = features["client_id"].astype(str)
unique_clients = np.array(clients.drop_duplicates())
rng = np.random.default_rng(42)
shuffled = rng.permutation(unique_clients)
test_clients = set(shuffled[: max(1, int(round(len(shuffled) * 0.2)))])
test_mask = clients.isin(test_clients).to_numpy()
train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

if len(train_idx) == 0 or len(test_idx) == 0:
    raise ValueError("Client holdout split produced empty train or test set.")

print(f"Rows: {len(features):,}")
print(f"Train rows: {len(train_idx):,}")
print(f"Test rows: {len(test_idx):,}")
print(f"Client-holdout split: {len(test_clients)} test clients")
print(f"Base rate in test set: {y.iloc[test_idx].mean():.3f}")


Rows: 30,000
Train rows: 27,675
Test rows: 2,325
Client-holdout split: 6 test clients
Base rate in test set: 0.391


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a client-aware 80/20 holdout split, grouped by `client_id`, not a random row split. That is the honest choice for this question because the model is meant to support refresh prioritization for unseen client portfolios, not just re-score pages already seen in the same account. A row-level split would leak client-specific patterns and make the model look stronger than it would be in deployment. I also keep the baseline and the learned model on the same train/test partition so the comparison is fair.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# The split is client-based to avoid leaking one client's pattern into another client's evaluation.

clients = features["client_id"].astype(str)
unique_clients = np.array(clients.drop_duplicates())
random_generator = np.random.default_rng(42)
shuffled_clients = random_generator.permutation(unique_clients)
cutoff = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:cutoff])
client_test_mask = clients.isin(test_clients).to_numpy()
train_idx = np.where(~client_test_mask)[0]
test_idx = np.where(client_test_mask)[0]

train_target = y.iloc[train_idx]
test_target = y.iloc[test_idx]

print(f"Train rows: {len(train_idx):,}")
print(f"Test rows: {len(test_idx):,}")
print(f"Test clients: {len(test_clients):,}")
print(f"Base rate in test set: {test_target.mean():.3f}")
print(f"Base rate in train set: {train_target.mean():.3f}")


Train rows: 27,675
Test rows: 2,325
Test clients: 6
Base rate in test set: 0.391
Base rate in train set: 0.555


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Same data, same split, same metric as the baseline: the model has to prove it beats the rule.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def precision_at_k(y_true, scores, k):
    frame = pd.DataFrame({"y": y_true, "score": scores})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0


X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "classifier",
        LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    ),
])
model.fit(X_train, y_train)

model_probs = model.predict_proba(X_test)[:, 1]
model_pred = (model_probs >= 0.5).astype(int)

baseline_lookup = baseline.set_index("content_id")["baseline_refresh_score"]
baseline_test_scores = (
    features.loc[test_idx, "content_id"].map(baseline_lookup).fillna(0.0).to_numpy()
)

baseline_metrics = {
    "method": "baseline",
    "accuracy": accuracy_score(y_test, (baseline_test_scores >= 0.5).astype(int)),
    "precision": precision_score(y_test, (baseline_test_scores >= 0.5).astype(int), zero_division=0),
    "recall": recall_score(y_test, (baseline_test_scores >= 0.5).astype(int), zero_division=0),
    "f1": f1_score(y_test, (baseline_test_scores >= 0.5).astype(int), zero_division=0),
    "precision_at_20": precision_at_k(y_test, baseline_test_scores, 20),
    "precision_at_50": precision_at_k(y_test, baseline_test_scores, 50),
    "precision_at_100": precision_at_k(y_test, baseline_test_scores, 100),
    "roc_auc": roc_auc_score(y_test, baseline_test_scores),
    "average_precision": average_precision_score(y_test, baseline_test_scores),
}

model_metrics = {
    "method": "logistic_regression",
    "accuracy": accuracy_score(y_test, model_pred),
    "precision": precision_score(y_test, model_pred, zero_division=0),
    "recall": recall_score(y_test, model_pred, zero_division=0),
    "f1": f1_score(y_test, model_pred, zero_division=0),
    "precision_at_20": precision_at_k(y_test, model_probs, 20),
    "precision_at_50": precision_at_k(y_test, model_probs, 50),
    "precision_at_100": precision_at_k(y_test, model_probs, 100),
    "roc_auc": roc_auc_score(y_test, model_probs),
    "average_precision": average_precision_score(y_test, model_probs),
}

comparison = pd.DataFrame([baseline_metrics, model_metrics]).round(3)
print(comparison.to_string(index=False))


             method  accuracy  precision  recall    f1  precision_at_20  precision_at_50  precision_at_100  roc_auc  average_precision
           baseline     0.609      0.499   0.189 0.274             0.15             0.24              0.36    0.627              0.468
logistic_regression     0.661      0.566   0.567 0.566             0.35             0.40              0.44    0.700              0.522


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The learned model looks strongest when the page is both visible and underperforming: high search demand, weak engagement, and stale content are the patterns that drive the probability upward. The main failure mode is when the page has conflicting signals — for example, a page that still ranks well but has one-off changes in traffic shape, or a newly published page that is briefly weak but not genuinely stale. Those are exactly the cases where a single logistic coefficient is too blunt. I checked the top coefficients and the hard errors, and the model is most sensitive to the same signals the baseline already tracks: low engagement, stale freshness, and declining demand.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Read the error surface: what the model leans on and where it gets confused.

coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coef": model.named_steps["classifier"].coef_[0],
})
coefficients["abs_coef"] = coefficients["coef"].abs()
print("Top 10 positive coefficients:")
print(coefficients.sort_values("coef", ascending=False).head(10)[["feature", "coef"]].to_string(index=False))
print("\nTop 10 negative coefficients:")
print(coefficients.sort_values("coef", ascending=True).head(10)[["feature", "coef"]].to_string(index=False))

error_frame = pd.DataFrame({
    "content_id": features.loc[test_idx, "content_id"].to_numpy(),
    "client_id": features.loc[test_idx, "client_id"].to_numpy(),
    "actual": y_test.to_numpy(),
    "predicted": model_pred,
    "probability": model_probs,
})
error_frame["is_wrong"] = error_frame["actual"] != error_frame["predicted"]

false_positives = error_frame[(error_frame["actual"] == 0) & (error_frame["is_wrong"])].sort_values("probability", ascending=False).head(3)
false_negatives = error_frame[(error_frame["actual"] == 1) & (error_frame["is_wrong"])].sort_values("probability", ascending=False).head(3)

print("\nHard false positives (actual=0, predicted=1):")
print(false_positives[["content_id", "client_id", "probability", "actual", "predicted"]].to_string(index=False))
print("\nHard false negatives (actual=1, predicted=0):")
print(false_negatives[["content_id", "client_id", "probability", "actual", "predicted"]].to_string(index=False))


Top 10 positive coefficients:
                     feature     coef
         log_impressions_90d 1.658755
                  word_count 1.607656
         impression_tier_low 0.245641
   word_count_tier_1000-2000 0.184402
      days_since_last_update 0.180911
         main_intent_unknown 0.119203
                 scroll_rate 0.116362
content_type_keyword article 0.094765
         freshness_tier_0-30 0.070423
              age_tier_31-90 0.066715

Top 10 negative coefficients:
                  feature      coef
               char_count -1.351979
           log_clicks_90d -0.656016
             avg_position -0.404586
    days_with_impressions -0.251329
         content_age_days -0.248888
         log_sessions_90d -0.243358
      position_tier_top_3 -0.190942
     impression_tier_good -0.179889
impression_tier_excellent -0.177047
word_count_tier_2000-3500 -0.121412

Hard false positives (actual=0, predicted=1):
          content_id         client_id  probability  actual  predicted
content

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.